# Analysis

In [18]:
import pandas as pd

data = pd.read_csv("../data/simulated_data.csv")

In [19]:
data.head()

,plant_id,group,concentration,start_height,end_height,growth
0,0,control,0.0,6.065775,10.059299,3.993523
1,1,control,0.0,4.568736,11.463168,6.894432
2,2,control,0.0,5.846615,9.488214,3.641600
3,3,control,0.0,3.113303,8.768249,5.654946
4,4,control,0.0,4.001840,9.840455,5.838615


## One-way ANOVA

In [20]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols("growth ~ C(group)", data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)

              sum_sq     df          F        PR(>F)
C(group)  411.949623    3.0  64.445988  3.671690e-27
Residual  332.392772  156.0        NaN           NaN


## Pairwise Tukey Test

This tells us which groups differ significantly.

In [21]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=data["growth"],
    groups=data["group"],
    alpha=0.05
)

print(tukey)

    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
   group1      group2   meandiff p-adj  lower  upper  reject
------------------------------------------------------------
    control treatment_1   1.8912   0.0  1.0436 2.7389   True
    control treatment_2   2.5409   0.0  1.6933 3.3886   True
    control treatment_3   4.4915   0.0  3.6439 5.3391   True
treatment_1 treatment_2   0.6497 0.196 -0.1979 1.4973  False
treatment_1 treatment_3   2.6003   0.0  1.7526 3.4479   True
treatment_2 treatment_3   1.9506   0.0  1.1029 2.7982   True
------------------------------------------------------------


## ANCOVA

This takes into account covariates that could also affect the final height of the plant (in this case, the starting height). This is probably the most robust method.

In [22]:
model = ols("end_height ~ C(group) + start_height", data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(anova_table)

                  sum_sq     df           F        PR(>F)
C(group)      503.224883    3.0  146.485836  4.908086e-45
start_height    0.237732    1.0    0.207608  6.492867e-01
Residual      177.491238  155.0         NaN           NaN


## Ordinary Least Squares Regression

This models the impact of the concentration on the final height.

We can interpret these (simulated) results as saying that since the coefficient for the concentration is ~0.43, for every time the concentration is increased by 1%, the final height is predicted to increase by 0.43cm. Note that these units are whatever is used in the data.

In [23]:
model = ols("growth ~ concentration", data=data).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 growth   R-squared:                       0.530
Model:                            OLS   Adj. R-squared:                  0.527
Method:                 Least Squares   F-statistic:                     178.4
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           1.01e-27
Time:                        22:15:56   Log-Likelihood:                -289.55
No. Observations:                 160   AIC:                             583.1
Df Residuals:                     158   BIC:                             589.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         5.0457      0.214     23.570